Getting Net Deposit

In [17]:
from sqlalchemy import func, select
from sqlalchemy import create_engine
from sqlalchemy.orm import Session, sessionmaker
from sqlmodel import SQLModel
import backend.config as config

from typing import List, Optional
import datetime
import pandas as pd

from backend.db.models.postgres import Transactions

In [2]:
_dsn = (
    f"postgresql+psycopg2://{config.POSTGRES_USER}:{config.POSTGRES_PASSWORD}"
    f"@{config.POSTGRES_HOST}:{config.POSTGRES_PORT}/{config.POSTGRES_DB}"
)

engine = create_engine(_dsn, pool_size=5, max_overflow=10)
session_factory: sessionmaker[Session] = sessionmaker(engine, expire_on_commit=False)


In [9]:
types = ['WITHDRAW', 'TRANSFER']
start_date = None
end_date = None

In [36]:
def get_transactions(
    session: Session,
    types: List,
    start_date: Optional[datetime.datetime] = None,
    end_date: Optional[datetime.datetime] = None,
    ):    
    stmt = select(Transactions).order_by(
        Transactions.date_time.desc()
    ).where(Transactions.type.in_(types))
    
    if "WITHDRAW" in types:
        stmt = stmt.where(Transactions.amount < 0)
    
    if "DEPOSIT" in types:
        stmt = stmt.where(Transactions.amount > 0)

    if start_date is not None:
        stmt = stmt.where(Transactions.date_time >= start_date)

    if end_date is not None:
        stmt = stmt.where(Transactions.date_time <= end_date)
    
    return session.execute(stmt).scalars().all()

In [46]:
with session_factory() as session:
    response = get_transactions(
        session,
        types=["DEPOSIT"],
        start_date=datetime.datetime(2024, 1, 1),
        end_date=datetime.datetime(2025, 12, 31),
    )

In [47]:
response

[Transactions(amount=2061.17, currency='GBP', date_time=datetime.datetime(2025, 12, 4, 18, 51, 46, 692000, tzinfo=datetime.timezone.utc), reference='21e09827-ae50-44c6-b3ac-36d083ce3bc2', type='DEPOSIT', saved_at=datetime.datetime(2026, 4, 14, 15, 42, 5, 414726, tzinfo=datetime.timezone.utc)),
 Transactions(amount=500.0, currency='GBP', date_time=datetime.datetime(2025, 11, 28, 7, 20, 57, 728000, tzinfo=datetime.timezone.utc), reference='3ec2d881-cb38-46d7-9e42-d33da4fb7370', type='DEPOSIT', saved_at=datetime.datetime(2026, 4, 14, 15, 42, 5, 414726, tzinfo=datetime.timezone.utc)),
 Transactions(amount=500.0, currency='GBP', date_time=datetime.datetime(2025, 10, 31, 7, 13, 42, 984000, tzinfo=datetime.timezone.utc), reference='0a87a40f-a1dc-477d-b97c-05f3d844e3a5', type='DEPOSIT', saved_at=datetime.datetime(2026, 4, 14, 15, 42, 5, 414726, tzinfo=datetime.timezone.utc)),
 Transactions(amount=750.0, currency='GBP', date_time=datetime.datetime(2025, 10, 12, 22, 21, 26, 401000, tzinfo=dateti

In [39]:
df = pd.DataFrame([r.model_dump() for r in response])
df['amount'].sum()


np.float64(-2037.5400000000002)

In [40]:
type(response[0].date_time)

datetime.datetime